# Notebook 10: Capstone -- Distributed ML Pipeline

## Why This Matters

This capstone notebook synthesizes everything in Series 39: GPU memory hierarchy, distributed collectives, DDP, FSDP, Ray Core, Ray Train, Ray Tune, Ray Data, and Ray Serve. We design a complete ML pipeline from data ingestion through model serving, with distributed training, hyperparameter optimization, and fault-tolerant serving. This is the architecture review blueprint for a mid-to-large scale ML production system.


In [ ]:
%pip install -q 'ray[default,train,tune,serve,data]' torch numpy matplotlib

In [ ]:
import ray
from ray import tune, train, serve
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig
from ray.tune import Tuner, TuneConfig
from ray.tune.schedulers import ASHAScheduler
from ray.tune.search.optuna import OptunaSearch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import time, os, tempfile

if ray.is_initialized():
    ray.shutdown()
ray.init(num_cpus=4, ignore_reinit_error=True)
print("Ray initialized. Resources:", ray.cluster_resources())
print("Ready.")

## 1. Pipeline Architecture

The full distributed ML pipeline:

```
[Object Store (S3/GCS)]
        |
        v
[Ray Data Pipeline]
  - read_parquet(s3://...)
  - map_batches(tokenize, compute_batch=True)
  - local_shuffle_buffer_size=10000
        |
        v
[Ray Train + FSDP]
  - TorchTrainer with ScalingConfig(num_workers=8, use_gpu=True)
  - FSDP wrapping for models > single GPU
  - Checkpoint every N steps
        |
        v
[Ray Tune + ASHA]
  - Bayesian search over lr, wd, scheduler
  - ASHA kills bottom 50% every 1000 steps
  - Parallel experiments
        |
        v
[Ray Serve]
  - Preprocessor deployment (CPU, 4 replicas)
  - Model deployment (GPU, 2 replicas)
  - Auto-scaling based on load
```


In [ ]:
# Define model architecture
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim, dropout):
        super().__init__()
        layers = []
        dims = [input_dim] + list(hidden_dims)
        for i in range(len(dims) - 1):
            layers += [
                nn.Linear(dims[i], dims[i+1]),
                nn.LayerNorm(dims[i+1]),
                nn.GELU(),
                nn.Dropout(dropout)
            ]
        layers.append(nn.Linear(dims[-1], output_dim))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

# Test model
model = MLPClassifier(64, [128, 64], 10, 0.1)
x = torch.randn(8, 64)
print('Model output shape:', model(x).shape)
print('Params:', sum(p.numel() for p in model.parameters()))

## 2. Distributed Training with Ray Train

A complete training loop using Ray Train's TorchTrainer, with proper checkpointing and metric reporting.


In [ ]:
# Complete Ray Train pipeline
def create_dataset():
    N = 5000
    np.random.seed(42)
    X = np.random.randn(N, 64).astype(np.float32)
    y = np.random.randint(0, 10, N)
    return X, y

def train_loop_fn(config):
    lr = config['lr']
    hidden = config['hidden']
    dropout = config['dropout']
    n_epochs = config.get('n_epochs', 3)
    batch_size = config.get('batch_size', 128)
    
    # Model + DDP wrap
    model = MLPClassifier(64, [hidden, hidden//2], 10, dropout)
    model = train.torch.prepare_model(model)
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    criterion = nn.CrossEntropyLoss()
    
    # Data
    X, y = create_dataset()
    split = int(0.8 * len(X))
    train_ds = TensorDataset(torch.from_numpy(X[:split]), torch.from_numpy(y[:split]))
    val_ds   = TensorDataset(torch.from_numpy(X[split:]), torch.from_numpy(y[split:]))
    
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl   = DataLoader(val_ds, batch_size=batch_size)
    train_dl = train.torch.prepare_data_loader(train_dl)
    val_dl   = train.torch.prepare_data_loader(val_dl)
    
    for epoch in range(n_epochs):
        model.train()
        train_losses = []
        for xb, yb in train_dl:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())
        scheduler.step()
        
        model.eval()
        val_losses, correct = [], 0
        with torch.no_grad():
            for xb, yb in val_dl:
                logits = model(xb)
                val_losses.append(criterion(logits, yb).item())
                correct += (logits.argmax(1) == yb).sum().item()
        val_acc = correct / len(val_ds)
        
        # Checkpoint
        with tempfile.TemporaryDirectory() as td:
            torch.save({'model': model.state_dict(), 'epoch': epoch},
                      os.path.join(td, 'ckpt.pt'))
            from ray.train import Checkpoint
            ckpt = Checkpoint.from_directory(td)
            train.report({'train_loss': np.mean(train_losses),
                          'val_loss': np.mean(val_losses),
                          'val_acc': val_acc},
                         checkpoint=ckpt)

# Run training
trainer = TorchTrainer(
    train_loop_per_worker=train_loop_fn,
    train_loop_config={'lr': 1e-3, 'hidden': 128, 'dropout': 0.1, 'n_epochs': 3},
    scaling_config=ScalingConfig(num_workers=2)
)
result = trainer.fit()
print(f'Training complete. Val accuracy: {result.metrics["val_acc"]:.4f}')

## 3. Hyperparameter Search with ASHA + Optuna

Wrap the training loop with Ray Tune to find the best hyperparameters automatically.


In [ ]:
# HPO: ASHA + Optuna Bayesian search
def hpo_train_loop(config):
    # Same as train_loop_fn but uses tune's config
    train_loop_fn(config)

# Wrap TorchTrainer for use with Tune
def train_fn_with_ray_train(config):
    # This is the function Ray Tune calls
    # We just run the per-worker loop directly (single worker for HPO)
    model = MLPClassifier(64, [config['hidden'], config['hidden']//2], 10, config['dropout'])
    optimizer = optim.AdamW(model.parameters(), lr=config['lr'])
    criterion = nn.CrossEntropyLoss()
    
    X, y = create_dataset()
    split = int(0.8 * len(X))
    Xt, yt = torch.from_numpy(X[:split]), torch.from_numpy(y[:split])
    Xv, yv = torch.from_numpy(X[split:]), torch.from_numpy(y[split:])
    
    for epoch in range(5):
        model.train()
        for i in range(0, len(Xt), 128):
            xb, yb = Xt[i:i+128], yt[i:i+128]
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()
        
        model.eval()
        with torch.no_grad():
            val_acc = (model(Xv).argmax(1) == yv).float().mean().item()
        tune.report({'val_acc': val_acc, 'epoch': epoch})

# Configure HPO
asha = ASHAScheduler(metric='val_acc', mode='max', grace_period=2, max_t=5)
optuna = OptunaSearch(metric='val_acc', mode='max')

tuner = Tuner(
    train_fn_with_ray_train,
    param_space={
        'lr':      tune.loguniform(1e-4, 1e-2),
        'hidden':  tune.choice([64, 128, 256]),
        'dropout': tune.uniform(0.0, 0.4),
    },
    tune_config=TuneConfig(num_samples=12, scheduler=asha, search_alg=optuna)
)

tune_results = tuner.fit()
best = tune_results.get_best_result(metric='val_acc', mode='max')
print(f'Best config: {best.config}')
print(f'Best val_acc: {best.metrics["val_acc"]:.4f}')

## 4. Distributed Training Decision Guide

| Scenario | Recommendation | Reasoning |
|----------|---------------|-----------|
| Model fits on 1 GPU, < 8 GPUs | DDP | Simple, no overhead |
| Model fits on 1 GPU, > 8 GPUs | DDP + gradient compression | Communication bottleneck |
| Model needs 2-8 GPUs (8B-70B) | FSDP (ZeRO-3) | Memory savings worth 3x comm overhead |
| Model needs 100+ GPUs (100B+) | FSDP + Tensor Parallelism | Pipeline + tensor hybrid |
| Inference only, 1 GPU | Single process | No distribution needed |
| Inference, high throughput | Ray Serve + batching | Queue batches, maximize GPU util |
| Multiple models, routing | Ray Serve pipeline | Chain preprocessor + models |

**Memory rules of thumb**:
- 1B params = ~2GB bfloat16 = ~16GB with Adam optimizer states
- A100 80GB can hold: 40B params (bfloat16 weights only)
- With FSDP on 8 x A100: 320B params (weights only)


## 5. Interview Preparation: Distributed Training Q&A

**Q: What is the difference between DDP and FSDP?**

A: DDP replicates the full model on every GPU and all-reduces gradients after each backward pass. FSDP (= ZeRO Stage 3) shards parameters, gradients, and optimizer states across GPUs. FSDP uses 3x more communication per step (all-gather + reduce-scatter vs single all-reduce) but reduces per-GPU memory by a factor equal to the number of GPUs. Use DDP when the model fits; use FSDP when it does not.

**Q: Explain ring all-reduce.**

A: Ring all-reduce organizes N GPUs in a ring. Phase 1 (reduce-scatter): each GPU sends one shard forward N-1 times, accumulating partial sums. Phase 2 (all-gather): each GPU broadcasts its fully-reduced shard N-1 times. Total communication per GPU: 2*(N-1)/N * P bytes, approximately 2P bytes regardless of N. This is optimal -- every byte is transferred exactly twice.

**Q: What is the arithmetic intensity of matrix multiplication?**

A: For an (M, K) times (K, N) matmul: 2*M*K*N FLOPs, (M*K + K*N + M*N) * 4 bytes. Intensity = 2MKN / (4*(MK+KN+MN)). For large square matrices (M=K=N=4096): ~341 FLOPs/byte. This is above A100's ridge point (~156 FLOPs/byte), so large matmuls are compute-bound.

**Q: When would you use Ray over PyTorch distributed?**

A: Ray is better for: (1) multi-node orchestration beyond a single training job, (2) heterogeneous workloads (HPO + training + serving), (3) fault tolerance with automatic checkpointing, (4) Python-native actors for parameter servers. PyTorch distributed is better when you have a single training job and want minimal overhead.


## Summary

### Series 39 Key Concepts

| Topic | Key insight |
|-------|-------------|
| GPU memory hierarchy | HBM bandwidth (2 TB/s) limits memory-bound ops; stay in L1/L2 via fusion |
| Ring all-reduce | O(P) communication per GPU regardless of GPU count |
| DDP | One all-reduce per step; gradient bucketing + overlap are key |
| FSDP | ZeRO-3: 8x less memory, 3x more communication; worth it when model doesn't fit |
| Ray Core | Tasks (stateless) + Actors (stateful) + Object store (zero-copy) |
| Ray Train | High-level DDP/FSDP abstraction with fault tolerance |
| Ray Tune | ASHA kills poor trials early; Bayesian guides exploration efficiently |
| Ray Data | Streaming, lazy, distributed data pipeline for pre-training scale |
| Ray Serve | Batching + pipelines + auto-scaling for production model serving |

**The meta-lesson**: Every performance problem in distributed ML has a root cause in one of three bottlenecks: compute (FLOPs), memory (capacity or bandwidth), or communication (network bandwidth or latency). The tools in this series address each bottleneck explicitly.
